In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import joblib
import time

In [ ]:
start_time = time.time()
real_df = pd.read_csv(
    '../datasets/online-retail-ii-cleaned.csv.zstd',
    compression='zstd',
    sep=';'
)

df = real_df.copy()

In [ ]:
print(df.info())

display(df.head())

print("Missing Values:")
display(df.isna().sum())

print("Duplicated Rows:", df.duplicated().sum())

display(df[df.duplicated()])

In [ ]:
df = df[~df.duplicated()]

print("Remaining duplicates:", df.duplicated().sum())

In [ ]:
df_clean = df[
    (~df['IsNegQty']) &
    (~df['IsNegPrice']) &
    (~df['IsQuantityOutlier']) &
    (~df['IsPriceOutlier']) &
    (~df['IsGuest']) &
    (~df['IsCanceled'])  # hanya transaksi sukses
].copy()

In [ ]:
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

In [ ]:
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency
    'InvoiceID': 'nunique',                                   # Frequency
    'TotalPrice': 'sum'                                       # Monetary
}).reset_index()

rfm.columns = ['CustomerID', 'Recency', 'Frequency', 'Monetary']

display(rfm.head())

In [ ]:
rfm_log = rfm[['Recency', 'Frequency', 'Monetary']].apply(np.log1p)

display(rfm_log.describe())

In [ ]:
sns.set_theme(style="whitegrid")

fig, ax = plt.subplots(2, 3, figsize=(18, 10))
features = ['Recency', 'Frequency', 'Monetary']

for i, col in enumerate(features):

    sns.histplot(rfm[col], kde=True, ax=ax[0, i], color='skyblue')
    ax[0, i].set_title(f'{col} Distribution')

    sns.boxplot(x=rfm[col], ax=ax[1, i], color='salmon')
    ax[1, i].set_title(f'{col} Outliers')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(18, 10))
features = ['Recency', 'Frequency', 'Monetary']

for i, col in enumerate(features):

    sns.histplot(rfm_log[col], kde=True, ax=ax[0, i], color='skyblue')
    ax[0, i].set_title(f'{col} Distribution')

    sns.boxplot(x=rfm_log[col], ax=ax[1, i], color='salmon')
    ax[1, i].set_title(f'{col} Outliers')

plt.tight_layout()
plt.show()

In [ ]:
scaler = RobustScaler()

rfm_scaled = scaler.fit_transform(rfm_log)

df_scaled = pd.DataFrame(
    rfm_scaled,
    columns=['Recency', 'Frequency', 'Monetary']
)

display(df_scaled.head())

In [ ]:
inertia = []
K = range(1, 11)

for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(df_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8,5))
plt.plot(K, inertia, 'bx-')
plt.xlabel('Jumlah Cluster (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()

In [ ]:
for k in range(2, 8):

    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(df_scaled)

    score = silhouette_score(df_scaled, labels)

    print(f"k={k}, silhouette={score:.3f}")

In [ ]:
k_optimal = 4

kmeans = KMeans(
    n_clusters=k_optimal,
    random_state=42,
    n_init=10
)

rfm['Cluster'] = kmeans.fit_predict(df_scaled) + 1

rfm_map = {
    1:"Bronze",
    2:"Silver",
    3:"Gold",
    4:"Platinum"
}

rfm['Cluster'] = rfm['Cluster'].map(rfm_map)
rfm['Cluster'] = pd.Categorical(rfm['Cluster'], categories=['Bronze', 'Silver', 'Gold', 'Platinum'], ordered=True)

display(rfm.head())

In [ ]:
cluster_analysis = rfm.groupby('Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': ['mean', 'count']
}).round(2).sort_index(ascending=False)

heatmap_data = cluster_analysis.drop(columns=('Monetary', 'count'))

heatmap_data.columns = ['Recency', 'Frequency', 'Monetary']

heatmap_norm = (heatmap_data - heatmap_data.min()) / (heatmap_data.max() - heatmap_data.min())

plt.figure(figsize=(10,6))

sns.heatmap(
    heatmap_norm,
    annot=heatmap_data,
    fmt='.2f',
    cmap='viridis',
    linewidths=.5
)

plt.title('Cluster Profiling')
plt.ylabel('Cluster ID')

plt.show()

plt.savefig(
    "../outputs/rfm_cluster_heatmap.png",
    dpi=300,
    bbox_inches='tight'
);

In [ ]:
from scipy.stats import kruskal

# 1. Define the metrics we want to validate
metrics = ['Recency', 'Frequency', 'Monetary']

print("--- Kruskal-Wallis Statistical Validation ---")

for metric in metrics:
    # Group the original (non-log) data by Cluster
    # We use rfm because Kruskal-Wallis handles skewed data naturally
    groups = [rfm[rfm['Cluster'] == c][metric] for c in rfm['Cluster'].unique()]
    
    stat, p_value = kruskal(*groups)
    
    print(f"\nMetric: {metric}")
    print(f"P-value: {p_value:.4e}")
    
    if p_value < 0.05:
        print(f"✅ Result: Statistically Significant. The clusters have distinct {metric} behaviors.")
    else:
        print(f"❌ Result: Not Significant. The clusters do not differ enough in {metric}.")

In [ ]:
import scikit_posthocs as sp

# 1. Run Dunn's test with Bonferroni correction for Monetary (your highest variance metric)
# This confirms if 'Silver' is a unique 'Whale' group or just lucky outliers.
dunn_monetary = sp.posthoc_dunn(
    rfm, 
    val_col='Monetary', 
    group_col='Cluster', 
    p_adjust='bonferroni'
)

# 2. Create a clean "Significant or Not" matrix
# True (1) means they are statistically different groups
sig_matrix = dunn_monetary < 0.05

print("--- Dunn's Post-Hoc Significance Matrix (Monetary) ---")
print("True = Statistically Different Clusters\n")

# Display with a heatmap style for easy reading in your notebook
display(sig_matrix.style.background_gradient(cmap='RdYlGn', axis=None))

In [ ]:
import plotly.express as px

# Define the colors
cluster_colors = {
    'Platinum': '#E5E4E2', 
    'Gold': '#FFD700',     
    'Silver': '#C0C0C0',   
    'Bronze': '#CD7F32'    
}

fig = px.scatter_3d(
    rfm,
    x='Recency',
    y='Frequency',
    z='Monetary',
    color='Cluster',
    color_discrete_map=cluster_colors,
    category_orders={"Cluster": ["Bronze", "Silver", "Gold", "Platinum"]},
    # Lowering opacity to 0.5 to see through the clusters
    opacity=0.5, 
    title='Customer Segmentation (RFM)'
)

# Apply the dark theme and refine point appearance
fig.update_layout(
    template='plotly_dark', 
    margin=dict(l=0, r=0, b=0, t=40)
)

# Optional: Add a small border to points to make them pop against the dark background
fig.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))

fig.show()

In [ ]:
rfm.to_csv(
    '../outputs/rfm_customer_segmentation.csv',
    index=False
)

print("File exported: rfm_customer_segmentation.csv")

In [ ]:
cluster_analysis.to_csv(
    '../outputs/rfm_cluster_profile.csv'
)

print("File exported: rfm_cluster_profile.csv")

In [ ]:
joblib.dump(kmeans, "../outputs/kmeans_rfm_model.pkl")

In [ ]:
joblib.dump(
    scaler,
    "../outputs/rfm_scaler.pkl"
)

print("Scaler saved")

In [ ]:
kmeans_loaded = joblib.load("../outputs/kmeans_rfm_model.pkl")
scaler_loaded = joblib.load("../outputs/rfm_scaler.pkl")

In [ ]:
import numpy as np
import pandas as pd

new_customer = pd.DataFrame({
    "Recency":[10],
    "Frequency":[5],
    "Monetary":[120]
})

In [ ]:
new_customer_log = np.log1p(new_customer)

new_customer_scaled = scaler_loaded.transform(new_customer_log)

In [ ]:
cluster = kmeans_loaded.predict(new_customer_scaled)

print("Customer cluster:", cluster[0])

In [ ]:
end_time = time.time()
total_time = end_time - start_time

print(f"Total runtime: {total_time:.2f} seconds")